In [1]:
import json

def preprocess_psychology_10k(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    processed_data = []
    for entry in data:
        processed_data.append({
            "instruction": entry.get("instruction", "Provide professional psychological advice."),
            "input": entry.get("input", ""),
            "output": entry.get("output", "")
        })
    
    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in processed_data:
            f.write(json.dumps(entry) + '\n')
    print(f"Finished Psychology-10K: {len(processed_data)} rows")

preprocess_psychology_10k('Psychology-10K.json', 'temp_psych.jsonl')

Finished Psychology-10K: 9846 rows


In [2]:
import json

def preprocess_combined_dataset(input_file, output_file):
    processed_data = []
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                entry = json.loads(line)
                processed_data.append({
                    "instruction": "Acting as a therapist, respond to the following patient concern.",
                    "input": entry.get("Context", ""),
                    "output": entry.get("Response", "")
                })
            except json.JSONDecodeError:
                continue # Skip empty or malformed lines
                
    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in processed_data:
            f.write(json.dumps(entry) + '\n')
    print(f"Finished combined_dataset: {len(processed_data)} rows")

preprocess_combined_dataset('combined_dataset.json', 'temp_combined.jsonl')

Finished combined_dataset: 3512 rows


In [3]:
import pandas as pd
import json

def preprocess_counsel_chat(input_file, output_file):
    df = pd.read_csv(input_file)
    processed_data = []
    
    for _, row in df.iterrows():
        processed_data.append({
            "instruction": "Respond as a compassionate mental health counselor.",
            "input": str(row['questionText']),
            "output": str(row['answerText'])
        })
        
    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in processed_data:
            f.write(json.dumps(entry) + '\n')
    print(f"Finished counsel_chat: {len(processed_data)} rows")

preprocess_counsel_chat('20220401_counsel_chat.csv', 'temp_counsel.jsonl')

Finished counsel_chat: 2775 rows


In [4]:
def merge_and_finalize(files, final_filename):
    final_list = []
    seen_inputs = set() # To prevent duplicates
    
    for file in files:
        with open(file, 'r', encoding='utf-8') as f:
            for line in f:
                entry = json.loads(line)
                # Deduplication logic
                if entry['input'] not in seen_inputs:
                    final_list.append(entry)
                    seen_inputs.add(entry['input'])

    with open(final_filename, 'w', encoding='utf-8') as f:
        for entry in final_list:
            f.write(json.dumps(entry) + '\n')
            
    print(f"Final Merged Dataset Created: {final_filename}")
    print(f"Total Unique Samples: {len(final_list)}")

files_to_merge = ['temp_psych.jsonl', 'temp_combined.jsonl', 'temp_counsel.jsonl']
merge_and_finalize(files_to_merge, 'therapy_dataset_final.jsonl')

Final Merged Dataset Created: therapy_dataset_final.jsonl
Total Unique Samples: 7616


In [5]:
# Quick check for duplicates in the largest file
import pandas as pd
df_psych = pd.read_json('Psychology-10K.json')
duplicates = df_psych.duplicated(subset=['input']).sum()
print(f"Duplicates within Psychology-10K alone: {duplicates}")

Duplicates within Psychology-10K alone: 3267
